# NoteSmith Playground
## Turning messy notes into clean sentences

This notebook is for experimenting with NoteSmith models and techniques.

In [ ]:
import sys
import os
sys.path.append('..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from models.transformer import TransformerModel
from models.lm import LanguageModel
from models.seq2seq import Seq2SeqModel
from preprocessing.tokenizer import SimpleTokenizer
from preprocessing.dataset import NoteProcessingDataset, TextDataset
from training.trainer import Trainer
from training.utils import set_seed, get_device
from inference.generate import TextGenerator

print("NoteSmith modules imported successfully!")

## 1. Setup and Configuration

In [ ]:
# Set random seed for reproducibility
set_seed(42)

# Get device
device = get_device()
print(f"Using device: {device}")

## 2. Sample Data

In [ ]:
# Sample messy notes and their clean versions
messy_notes = [
    "mtg tmrw 3pm discuss proj",
    "buy milk eggs bread 2day",
    "call john re meeting setup",
    "dentist appt tue 2pm",
    "finish report by fri deadline",
    "email team about update",
    "book flight to nyc next wk",
    "review code changes b4 merge"
]

clean_sentences = [
    "Meeting tomorrow at 3pm to discuss project",
    "Buy milk, eggs, and bread today",
    "Call John regarding meeting setup",
    "Dentist appointment on Tuesday at 2pm",
    "Finish report by Friday deadline",
    "Email team about the update",
    "Book flight to NYC next week",
    "Review code changes before merge"
]

print(f"Sample data: {len(messy_notes)} note pairs")
for i, (messy, clean) in enumerate(zip(messy_notes[:3], clean_sentences[:3])):
    print(f"{i+1}. '{messy}' -> '{clean}'")

## 3. Tokenizer Experimentation

In [ ]:
# Initialize and train tokenizer
tokenizer = SimpleTokenizer(vocab_size=1000)

# Build vocabulary from all texts
all_texts = messy_notes + clean_sentences
tokenizer.build_vocab(all_texts)

print(f"Tokenizer vocabulary size: {len(tokenizer)}")
print(f"Special tokens: {tokenizer.special_tokens}")

# Test tokenization
sample_text = messy_notes[0]
encoded = tokenizer.encode(sample_text)
decoded = tokenizer.decode(encoded)

print(f"\nOriginal: '{sample_text}'")
print(f"Encoded: {encoded}")
print(f"Decoded: '{decoded}'")

## 4. Model Experimentation

In [ ]:
# Initialize different models
vocab_size = len(tokenizer)

# Language Model
lm_model = LanguageModel(vocab_size, embed_size=128, hidden_size=256)
print(f"Language Model parameters: {sum(p.numel() for p in lm_model.parameters()):,}")

# Transformer Model
transformer_model = TransformerModel(vocab_size, d_model=128, nhead=4, num_layers=2)
print(f"Transformer parameters: {sum(p.numel() for p in transformer_model.parameters()):,}")

# Seq2Seq Model
seq2seq_model = Seq2SeqModel(vocab_size, vocab_size, hidden_size=128)
print(f"Seq2Seq parameters: {sum(p.numel() for p in seq2seq_model.parameters()):,}")

## 5. Dataset Preparation

In [ ]:
# Create dataset
dataset = NoteProcessingDataset(
    messy_notes=messy_notes,
    clean_sentences=clean_sentences,
    tokenizer=tokenizer,
    max_length=64
)

print(f"Dataset size: {len(dataset)}")

# Sample from dataset
sample_input, sample_target = dataset[0]
print(f"Sample input shape: {sample_input.shape}")
print(f"Sample target shape: {sample_target.shape}")
print(f"Sample input: {sample_input[:10]}")
print(f"Sample target: {sample_target[:10]}")

## 6. Text Generation Experimentation

In [ ]:
# Test text generator with untrained model
generator = TextGenerator(lm_model, tokenizer, device)

# Test input
test_input = "mtg tmrw"

try:
    # Greedy generation
    greedy_output = generator.generate_greedy(test_input, max_length=20)
    print(f"Greedy: '{test_input}' -> '{greedy_output}'")
    
    # Sampling generation
    sampling_output = generator.generate_sampling(test_input, max_length=20, temperature=0.8)
    print(f"Sampling: '{test_input}' -> '{sampling_output}'")
    
except Exception as e:
    print(f"Generation error (expected with untrained model): {e}")
    print("This is normal for an untrained model!")

## 7. Training Setup Exploration

In [ ]:
# Setup training components
from torch.utils.data import DataLoader

# Create data loader
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Setup training components
model = lm_model
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss(ignore_index=tokenizer.vocab['<PAD>'])

# Create trainer
trainer = Trainer(model, optimizer, criterion, device)

print("Training setup complete!")
print(f"Model: {model.__class__.__name__}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Criterion: {criterion.__class__.__name__}")

## 8. Visualization and Analysis

In [ ]:
# Analyze text lengths
messy_lengths = [len(tokenizer.encode(text)) for text in messy_notes]
clean_lengths = [len(tokenizer.encode(text)) for text in clean_sentences]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(messy_lengths, bins=10, alpha=0.7, label='Messy Notes')
plt.hist(clean_lengths, bins=10, alpha=0.7, label='Clean Sentences')
plt.xlabel('Token Length')
plt.ylabel('Frequency')
plt.title('Text Length Distribution')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(messy_lengths, clean_lengths, alpha=0.7)
plt.xlabel('Messy Note Length')
plt.ylabel('Clean Sentence Length')
plt.title('Length Correlation')
plt.plot([0, max(messy_lengths)], [0, max(messy_lengths)], 'r--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Average messy note length: {np.mean(messy_lengths):.1f} tokens")
print(f"Average clean sentence length: {np.mean(clean_lengths):.1f} tokens")

## 9. Next Steps

This playground notebook demonstrates the basic NoteSmith architecture. To continue:

1. **Data Collection**: Gather more messy note / clean sentence pairs
2. **Model Training**: Train models on the collected data
3. **Evaluation**: Implement evaluation metrics (BLEU, ROUGE, etc.)
4. **Fine-tuning**: Experiment with different architectures and hyperparameters
5. **Deployment**: Create a user-friendly interface for note cleaning

In [ ]:
# Save tokenizer for future use
tokenizer.save('../results/demo_tokenizer.pkl')
print("Tokenizer saved to results/demo_tokenizer.pkl")